In [2]:
!pip install \
evaluate \
bert-score \
#bleurt \
sacrebleu \
huggingface_hub \
camel_tools \
textstat \
nltk

  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached huggingface_hub-1.13.0-py3-none-any.whl (660 kB)
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.17.3
    Uninstalling huggingface-hub-0.17.3:
      Successfully uninstalled huggingface-hub-0.17.3
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/5 [huggingface-hub]
    Found existing installation: tokenizers 0.14.10/5

In [1]:
!pip install camel_tools

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 4.1 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 59.5 MB/s  0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13782 sha256=35350358661bb1220e6a764e6f3788269aacd7aa9eb9c7c0573d966302184b5d
  Stored in directory: /teamspace/studios/this_studio/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [camel_tools] [future]


In [3]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import camel_tools 
from camel_tools.utils.normalize import normalize_unicode
#to normalize 'alef' variants
from camel_tools.utils.normalize import normalize_alef_maksura_ar
#to normalize 'ya2' variants
from camel_tools.utils.normalize import normalize_alef_ar
#to normalize 'teh marbota' variants
from camel_tools.utils.normalize import normalize_teh_marbuta_ar
#to remove diacritical marks
from camel_tools.utils.dediac import dediac_ar
#import textstat
import time
import os
from huggingface_hub import HfApi, hf_hub_download

In [56]:
df = pd.read_csv("gemma-3n-E4B_Curr_Egy_outputs_fixed.csv")

In [57]:
df.head()

,lecture,image_name,output
0,CHI-91486,CHI-91486-0010000.jpg,عايز اشرحلك الشريحه دي؟ الشريحه دي عنوانها Cro...
1,CHI-91486,CHI-91486-0047450.jpg,عايزين نتكلم عن الخلفيه اللي احنا بنحاول نفهمه...
2,CHI-91486,CHI-91486-0077450.jpg,عايز اشرحلك الشريحه دي اللي بتتكلم عن التكيف ا...
3,CHI-91486,CHI-91486-0111450.jpg,عايز اشرحلك الشريحه دي اللي بتتكلم عن ال Crowd...
4,CHI-91486,CHI-91486-0123450.jpg,عايز اشرحلك الصوره دي اللي فيها شرح لعمليه الت...


In [58]:
#cleaning before evaluation 
def clean(df , col):
    '''
    The input should be the df itself + the column name that contains the text data.
    '''
    #df = df.drop_duplicates(subset=[col])
    #df = df.dropna(subset=[col])

    
    df = df.reset_index(drop=True)
    df[col] = df[col].fillna("").astype(str)
    for i in range(len(df)):
        sentence = df.loc[i, col]
        sentence = normalize_unicode(sentence)
        sentence = dediac_ar(sentence)
        sentence = normalize_alef_maksura_ar(sentence)
        sentence = normalize_alef_ar(sentence)
        sentence = normalize_teh_marbuta_ar(sentence)
        #instead of removing english we keep it so its penalized for english usage
        #this regex will remove all non-Arabic letters and non-space characters
        #sentence = re.sub(r'[^\u0600-\u06FF\s]', '', sentence, flags=re.UNICODE)
        df.loc[i , col] = sentence
    return df  

In [59]:
df.columns

Index(['lecture', 'image_name', 'output'], dtype='object')

In [60]:
df = clean(df , 'output')

In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lecture     1067 non-null   object
 1   image_name  1067 non-null   object
 2   output      1067 non-null   object
dtypes: object(3)
memory usage: 25.1+ KB


In [62]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1114 entries, 0 to 1113
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   lecture  1114 non-null   object
 1   image    1114 non-null   object
 2   source   1114 non-null   object
 3   aya      1114 non-null   object
 4   qwen     1114 non-null   object
 5   ain      1114 non-null   object
 6   gemma    1114 non-null   object
dtypes: object(7)
memory usage: 61.1+ KB


In [63]:
df = df.rename(columns = {'image_name' : 'image'})

In [64]:
df = df.merge(b[['image', 'source']], on='image', how='left')

In [65]:
df

,lecture,image,output,source
0,CHI-91486,CHI-91486-0010000.jpg,عايز اشرحلك الشريحه دي؟ الشريحه دي عنوانها Cro...,أهلاً، وشكراً لاهتمامكم بورقتنا البحثية عن الا...
1,CHI-91486,CHI-91486-0047450.jpg,عايزين نتكلم عن الخلفيه اللي احنا بنحاول نفهمه...,محتوى النصوص في الواقع المعزز (AR). الواقع الم...
2,CHI-91486,CHI-91486-0077450.jpg,عايز اشرحلك الشريحه دي اللي بتتكلم عن التكيف ا...,السياق المادي. وعشان نخلي بحثنا ملموس، ركزنا ع...
3,CHI-91486,CHI-91486-0111450.jpg,عايز اشرحلك الشريحه دي اللي بتتكلم عن ال Crowd...,دراستهم القايمة على المعمل. وعشان نسرع وضع إرش...
4,CHI-91486,CHI-91486-0123450.jpg,عايز اشرحلك الصوره دي اللي فيها شرح لعمليه الت...,الهدف هو جمع ملاحظات المستخدمين على مجموعة متن...
...,...,...,...,...
1062,CHI-214E5,CHI-214E5-0215000.jpg,عايزين نتكلم عن الكي لوينج كومبيتينسيز اللي هي...,اختبرنا الطلاب باستخدام اختبارات قبلية وبعدية،...
1063,CHI-214E5,CHI-214E5-0231450.jpg,عايزين نشوف ايه الفرق بين Zoom و RobotAR في تق...,أكملوا خاصتهم. وبالمثل، باستخدام الواقع المعزز...
1064,CHI-214E5,CHI-214E5-0244450.jpg,عن التعليقات اللي في الردود في الردود في الردو...,فدي كانت أبرز النقط من دراسات المستخدمين دي. م...
1065,CHI-214E5,CHI-214E5-0260450.jpg,الاساتذه كان عندهم وصول احسن ل مكان الطلاب و م...,المدربين كمان بلغوا إنهم بقوا قادرين يقدموا تو...


In [20]:
b = pd.read_csv('baseline_outputs.csv')
b.head()

,lecture,image,source,aya,qwen,ain,gemma
0,CHI-01C34,CHI-01C34-0339000.jpg,الأساس المنطقي للحدود دي. وبالنيابة عن زملائي ...,عندما وقت الشاشة ليس وقت الشاشة\n\nتوترات واحت...,"هذا الشرائح يتكلم عن موضوع: ""عندما يكون وقت ال...",أهلاً بكم في هذه الشريحة عن وقت الشاشة. إنها ع...,أهلاً بك يا بني! هذه الشريحة من عرض تقديمي. خل...
1,CHI-01C34,CHI-01C34-0169000.jpg,أثناء استخدام التطبيق. برز موضوعين شاملين في ن...,هذه الشريحة تعرض نتائج دراسة حول استخدام تطبيق...,هذا الشرائح بيقول:\n\n**نتائج الدراسة**\n\n(1)...,مرحباً يا طلاب! دعنا نتحدث عن هذه الشريحة.\n\n...,أهلاً بك يا بني! هذه الشريحة من عرض نتائج دراس...
2,CHI-01C34,CHI-01C34-0020000.jpg,أهلاً، اسمي صبا قواس وأنا طالبة دكتوراه في جام...,عندما وقت الشاشة ليس وقت الشاشة: التوترات والا...,"هذا الشرائح بعنوان: ""عندما يكون وقت الشاشة ليس...",أهلاً بكم، أصدقائي! دعونا نتحدث عن هذه الشريحة...,أهلاً بك يا بني! هذه شريحة من عرض تقديمي. دعني...
3,CHI-01C34,CHI-01C34-0126000.jpg,توترات. سألنا السؤال التالي: إزاي الأهالي بيخت...,هذه الشريحة تتناول أسئلة البحث المتعلقة بتطبيق...,"هذا الشرائح بعنوان ""أسئلة البحث"" وده مكتوب بخط...",مرحبا يا طلاب! دعنا نتحدث عن هذه الشريحة.\n\nا...,أهلاً بك يا بني! هذه الشريحة تتحدث عن أسئلة ال...
4,CHI-01C34,CHI-01C34-0207000.jpg,استخدام التكنولوجيا كمرحلة انتقالية. أهالي كتي...,عائلات' تجارب استخدام تطبيق تينز خلال استكشاف ...,هذا الشرائح عن تجربة العائلات في استخدام تطبيق...,مرحبا يا طلاب! دعنا نتحدث عن هذه الشريحة عن تج...,أهلاً بك يا بني/يا بنتي، سأشرح لك هذه الشريحة ...


In [44]:
import evaluate
from bert_score import score
import numpy as np
import bert_score


meteor = evaluate.load("meteor")       # METEOR
bleu = evaluate.load("bleu")           # BLEU
chrf = evaluate.load("chrf")           # CHRF
bleurt = evaluate.load("bleurt")       # BLEURT

[nltk_data] Downloading package wordnet to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /teamspace/studios/this_studio/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


I0000 00:00:1777645221.739590   89326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777645221.949104   89326 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777645223.199351   89326 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Using default checkpoint 'bleurt-base-128' for sequence maximum

INFO:tensorflow:Reading checkpoint /teamspace/studios/this_studio/.cache/huggingface/metrics/bleurt/default/downloads/extracted/c9567476201aa8527ea010c5ce7f3b534bd6d445678239cb4a0c4147530e5f31/bleurt-base-128.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint bert_custom
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:bert_custom
INFO:tensorflow:... vocab_file:vocab.txt
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... do_lower_case:True
INFO:tensorflow:... max_seq_length:128
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating WordPiece tokenizer.
INFO:tensorflow:WordPiece tokenizer instantiated.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.


W0000 00:00:1777645241.414816   89326 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [43]:
#!pip install nltk
#!pip install sacrebleu
#!pip install git+https://github.com/google-research/bleurt.git

  Cloning https://github.com/google-research/bleurt.git to /tmp/pip-req-build-j2hr4q0p
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/bleurt.git /tmp/pip-req-build-j2hr4q0p

  Resolved https://github.com/google-research/bleurt.git to commit cebe7e6f996b40910cfaa520a63db47807e3bf5c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 91.4 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 11.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 265.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 139.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 141.6 MB/s  0:00:00
  Created wheel for BLEURT: filename=bleurt-0.0.2-py3-none-any.whl size=16456845 sha256=c363de80085229999a5b3f5584c6545aca95e9890c3d0edf

In [66]:
references = df['source'].tolist()
predictions = df["output"].tolist()

In [45]:
# computationally expenisive to an extent
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "AMR-KELEG/Sentence-ALDi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def compute_aldi(predictions):
    sum = 0
    for p in predictions:
        inputs = tokenizer(p, return_tensors="pt")
        outputs = model(**inputs)
        logits = outputs.logits
        sum += min(max(0, logits[0][0].item()), 1)
    mean  = sum / len(predictions)
    return mean

config.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/651M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/651M [00:00<?, ?B/s]

In [67]:
meteor_score = meteor.compute(predictions=predictions, references=references)
print("METEOR:", meteor_score)

bleu_score = bleu.compute(predictions=predictions, references=references)
print("BLEU:", bleu_score)

chrf_score = chrf.compute(predictions=predictions, references=references)
print("CHRF+:", chrf_score)


bleurt_score = bleurt.compute(predictions=predictions, references=references)
print("BLEURT:", np.mean(bleurt_score['scores']))

P, R, F1 = bert_score.score(predictions, references, lang="ar", verbose=True)

print("Bert score Precision:", P.mean().item())
print("Bert score Recall:", R.mean().item())
print("Bert score F1:", F1.mean().item())

METEOR: {'meteor': 0.04257492785950839}
BLEU: {'bleu': 0.002498637961256573, 'precisions': [0.05244224522631371, 0.005399984571472653, 0.0008627586315079453, 0.00015953288770480034], 'brevity_penalty': 1.0, 'length_ratio': 1.1792439321257249, 'translation_length': 65882, 'reference_length': 55868}
CHRF+: {'score': 13.168198422533294, 'char_order': 6, 'word_order': 0, 'beta': 2}
BLEURT: -0.28662288350235554


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/34 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/17 [00:00<?, ?it/s]

done in 2.64 seconds, 404.80 sentences/sec
Bert score Precision: 0.6200234293937683
Bert score Recall: 0.6333864331245422
Bert score F1: 0.6245044469833374


In [68]:
#Readability
import textstat
scores = [textstat.osman(p) for p in predictions]
print("Readability score: " , np.mean(scores))

Readability score:  44.96419532216988


In [69]:
ALDI = compute_aldi(predictions)
print("ALDI score:" , ALDI)

ALDI score: 0.4664724903583139


In [51]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [textstat]
